# Learning lab — Jhabua screening, implemented by hand

**Companion to `learning.md`** (the Jhabua GIS Training Manual). This notebook is
where you *do* what the manual teaches. Each section matches a manual module:

| Notebook section | Manual lesson | What you implement by hand |
|---|---|---|
| 1. Setup + grid contract | 1.1, 3.2 | config, pixel area, grid inventory |
| 2. CRS + vector discipline | 2.2, 3.1 | reproject the boundary, measure area |
| 3. Raster anatomy | 2.1 | class distribution inside the district |
| 4. Rejecter masks by hand | 8.1 | build every Stage-A/Stage-B rejecter |
| 5. First-fails-wins classifier | 8.2 | reproduce the stages of `reason_codes.tif` |
| 6. Small-patch logic | 8.3 | `scipy.ndimage.label`, 1.0 ha threshold |
| 7. Confidence flags | 8.2 (flags) | shrubland / old-loss / cropland-kept flags |
| 8. Cartography from real data | 9.1 | RGBA render + the `extent` bug |
| 9. Scaling | 11.1 | windowed reads, `out_shape` |
| 10. Capstone checks | 12.1 | the assert battery |

Three rules for this notebook:
1. **Type, don't paste.** The goal is your hand-built result matching the real one.
2. Every section ends with a `✔` check against the real pipeline output. If a check
   fails, **that failure is the lesson** — read the diff before touching anything.
3. Read `src/` *after* you've written your own version. Only then can you tell
   whether you disagreed with a *decision* or with an *implementation*.

Nothing here writes into `data/`. Scratch maps go to `outputs/learning_lab/`.

## 1. Setup and the grid contract *(manual §4.1, §6.1)*

In [ ]:
from pathlib import Path
import numpy as np
import rasterio
import geopandas as gpd
import yaml

ROOT     = Path.cwd() if (Path.cwd() / "config").exists() else Path.cwd().parent
CFG_PATH = ROOT / "config" / "jhabua.yaml"
ALIGNED  = ROOT / "data" / "processed" / "aligned"
OUTDIR   = ROOT / "outputs" / "jhabua_agroforestry_screening"
LAB      = ROOT / "outputs" / "learning_lab"
LAB.mkdir(parents=True, exist_ok=True)

cfg = yaml.safe_load(CFG_PATH.read_text())
RES = float(cfg["grid"]["resolution_m"])
print("root:", ROOT)
print("configured resolution:", RES, "m")

### 1.1 — The aligned inventory

Confirm the contracted geometry: **crs, transform, shape** identical across layers.
`dtype` and `nodata` legitimately differ (classes are uint8; indices are float32).

In [ ]:
files = sorted(ALIGNED.glob("*.tif"))
contracts = {}
for f in files:
    with rasterio.open(f) as src:
        contracts[f.name] = (src.crs, src.transform, src.shape)
        print(f"{f.name:28s} {contracts[f.name][0]}  shape={contracts[f.name][2]}")

# ✔ CHECK — the real load_layers() treats this same equality as a hard error
uniq = set(contracts.values())
assert len(uniq) == 1, f"grid contract drifted: {len(uniq)} distinct contracts"
print(f"\n✔ all {len(files)} aligned layers share one grid contract")

### 1.2 — Pixel area, derived not typed *(manual §3.2)*

In [ ]:
with rasterio.open(ALIGNED / "district_mask.tif") as src:
    a, b, c, d, e, f = tuple(src.transform)[:6]
    px_w, px_h = abs(a), abs(e)

px_ha = px_w * px_h / 10_000.0
print(f"pixel: {px_w} m × {px_h} m = {px_ha:.2f} ha")

# ✔ CHECK
assert abs(px_ha - 0.09) < 1e-9
assert b == 0 and d == 0, "rotated/sheared transform not supported (same rule as screening.py)"
print("✔ 1 px = 0.09 ha, derived from the affine transform")

## 2. CRS discipline and vector truth *(manual §3.1, §4.2)*

The manual's constant: **3,433.78 km²**. Measure it yourself after *explicit*
reprojection — and see what EPSG:4326 does to the same geometry.

In [ ]:
district = gpd.read_file(ROOT / "data" / "processed" / "boundary_jhabua.geojson")
utm = district.to_crs("EPSG:32643")
area_km2 = utm.area.iloc[0] / 1e6
print(f"area (EPSG:32643): {area_km2:,.2f} km² = {area_km2 * 100:,.1f} ha")
print("bounds:", utm.total_bounds.round(1))

wgs = district.to_crs("EPSG:4326")
print(f"area in square degrees (4326): {wgs.area.iloc[0]:,.4f}  ← meaningless in a report")

# ✔ CHECK
assert abs(area_km2 - 3433.78) < 0.02, f"got {area_km2:.2f} km², expected 3433.78"
print("✔ district area matches the waterfall's district total")

## 3. Raster anatomy — class distribution inside the district *(manual §2.1)*

WorldCover classes in Jhabua: **10 (tree), 20 (shrub), 30 (grass), 40 (cropland),
50 (built), 60 (bare), 80 (water)**, plus nodata/outside.

In [ ]:
with rasterio.open(ALIGNED / "district_mask.tif") as src:
    inside = src.read(1).astype(bool)

with rasterio.open(ALIGNED / "worldcover_2021.tif") as src:
    wc = src.read(1)

vals, counts = np.unique(wc[inside], return_counts=True)
DISTRICT_PX = counts.sum()
names = {10: "tree cover", 20: "shrubland", 30: "grassland", 40: "cropland",
         50: "built-up", 60: "bare/sparse", 80: "water"}
print(f"district pixels: {DISTRICT_PX:,}")
for v, c in zip(vals, counts):
    print(f"class {int(v):3d} {names.get(int(v), "other"):>13s}: {c:>10,} px  ({c / DISTRICT_PX:6.1%})")

# ✔ CHECK — two facts the manual quotes
assert DISTRICT_PX == 3_815_340, f"got {DISTRICT_PX:,}"
assert counts[vals == 40][0] == max(counts), "cropland (40) should dominate"
print("✔ 3,815,340 district px; cropland dominates → census-mode candidate headline")

## 4. Rejecter masks, hand-built *(manual §8.1, §8.2)*

The real classifier = **first failing filter wins**, per stage:

**Stage A (methodology):** `NODATA → WOODY_COVER → RECENT_WOODY_REMOVAL`
**Stage B (design):** `BUILTUP_WATER_WETLAND → CROPLAND → PROTECTED → STEEP_SLOPE`
**After both stages:** small eligible patches are re-coded **8** (transaction screen).

### 4.0 — Load the aligned layers, keeping each layer's nodata

In [ ]:
s = cfg["screening"]
d = s["design_screens"]
w = s["woody"]
start = cfg["project"]["start_year"]
print("start_year:", start, "| woody source:", w["source"],
      "| threshold:", w["threshold_pct"], "% | shrubland:", w["shrubland"])

L, masked = {}, {}        # raw arrays / per-layer nodata masks
for key, fname in cfg["paths"]["layers"].items():
    path = ALIGNED / fname
    if not path.exists():
        continue
    with rasterio.open(path) as src:
        arr = src.read(1)
        m = np.zeros(arr.shape, dtype=bool)
        if src.nodata is not None:
            m |= (arr == src.nodata)
            if arr.dtype.kind == "f":
                m |= ~np.isfinite(arr)
        L[key], masked[key] = arr, m
print("loaded:", ", ".join(sorted(L)))

### 4.1 — Stage-A rejecters *(methodology + data validity)*

- `NODATA` — any **core** layer unreadable for a pixel (the real pipeline unions
  each layer's rasterio mask; your next cell mirrors that exactly).
- `WOODY_COVER` — pre-existing woody ≥ 10%. Three proxy rules in one:
  (a) WorldCover tree classes (10, 95); (b) Hansen treecover ≥ 10% on **canopy now**
  (treecover set to 0 where loss predates start year); (c) Hansen gain areas if
  treated as woody. `source: either` = conservative OR.
- `RECENT_WOODY_REMOVAL` — Hansen loss younger than 10 years on pixels that had
  canopy. In Jhabua this is genuinely **zero** — a measured zero is data, not absence.

In [ ]:
NODATA_LAYERS = ("worldcover", "hansen_treecover2000", "hansen_lossyear",
                 "hansen_gain", "slope_deg", "protected_areas")

nodata = np.zeros(inside.shape, dtype=bool)
for key in NODATA_LAYERS:
    nodata |= masked[key]
nodata &= inside            # validity is asked inside the district only

wc_woody = np.isin(L["worldcover"], w["worldcover_classes"])
# careful: cast BEFORE adding 2000, or the eager `2000 + uint8` overflows (teachable!)
loss_year = np.where(L["hansen_lossyear"] > 0,
                     L["hansen_lossyear"].astype(np.int32) + 2000, 0)
lost_pre = (loss_year > 0) & (loss_year <= start)
canopy_now = np.where(lost_pre, 0, L["hansen_treecover2000"])
han_woody = canopy_now >= w["threshold_pct"]
if w["treat_hansen_gain_as_woody"]:
    han_woody = han_woody | L["hansen_gain"].astype(bool)
woody = wc_woody | han_woody      # source: either → conservative OR

had_canopy = L["hansen_treecover2000"] >= w["threshold_pct"]
age = start - loss_year
recent_removal = ((loss_year > 0) & (loss_year <= start)
                  & (age <= s["recent_removal_years"]) & had_canopy)

for label, m in [("NODATA", nodata), ("WOODY_COVER", woody),
                 ("RECENT_WOODY_REMOVAL", recent_removal)]:
    n = int((m & inside).sum())
    print(f"{label:22s} {n:>10,} px  ({n * px_ha:,.2f} ha)")

### 4.2 — Stage-B rejecters *(design choices, not methodology)*

**Read the numbers, then the lesson below the table.** Raw mask counts are *supersets*
of the waterfall rows, because the waterfall attributes each pixel to the **first**
rule that rejects it — overlap pixels are claimed by an earlier rule.

In [ ]:
import csv

built_water = np.isin(L["worldcover"], d["builtup_water_wetland_classes"])
cropland = (L["worldcover"] == d["cropland_worldcover_class"]) \
    if d["exclude_cropland"][cfg["project"]["eligibility_mode"]] \
    else np.zeros_like(built_water)
protected = L["protected_areas"].astype(bool)
steep = L["slope_deg"] > d["max_slope_deg"]

wf = {r["step"]: float(r["removed_ha"])
      for r in csv.DictReader((OUTDIR / "waterfall.csv").open())}

print(f"{'rule':22s} {'raw mask ha':>12s} {'waterfall ha':>13s} {'difference ha':>14s}")
for label, mask in [("BUILTUP_WATER_WETLAND", built_water), ("CROPLAND", cropland),
                    ("PROTECTED", protected), ("STEEP_SLOPE", steep)]:
    raw = int((mask & inside).sum()) * px_ha
    ref = wf[label]
    print(f"{label:22s} {raw:>12,.2f} {ref:>13,.2f} {raw - ref:>+14,.2f}")

# ✔ CHECK — raw masks must be SUPERSETS of the waterfall's attributed rows
for name, mask in [("BUILTUP_WATER_WETLAND", built_water),
                   ("PROTECTED", protected), ("STEEP_SLOPE", steep)]:
    raw = int((mask & inside).sum()) * px_ha
    assert raw >= wf[name] - 1.0, f"{name}: raw mask smaller than waterfall claim"
    print(f"✔ {name}: raw {raw:,.2f} ≥ attributed {wf[name]:,.2f} (overlap absorbed earlier)")
print("✔ every raw rejecter is the superset of its waterfall row —")
print("  the difference is overlap claimed by rules earlier in stage order.")

## 5. Reproduce the staged classifier *(manual §7.2, §8.2)*

**First failing filter wins.** One code per pixel — no pixel ever carries two reasons.
When two rules both want a pixel, the earliest in stage order claims it; the order
is a documented decision, not an accident.

In [ ]:
NAME2CODE = {"ELIGIBLE": 0, "WOODY_COVER": 1, "BUILTUP_WATER_WETLAND": 2,
             "CROPLAND": 3, "RECENT_WOODY_REMOVAL": 4, "STEEP_SLOPE": 5,
             "PROTECTED": 6, "NODATA": 7, "SMALL_PATCH": 8}
OUTSIDE = 255

rejecters = {"NODATA": nodata, "WOODY_COVER": woody,
             "RECENT_WOODY_REMOVAL": recent_removal,
             "BUILTUP_WATER_WETLAND": built_water, "CROPLAND": cropland,
             "PROTECTED": protected, "STEEP_SLOPE": steep}

reason = np.full(inside.shape, OUTSIDE, dtype=np.uint8)
reason[inside] = 0                                # everything inside starts eligible
for name in list(s["stage_a_order"]) + list(s["stage_b_order"]):
    newly = rejecters[name] & (reason == 0)
    reason[newly] = NAME2CODE[name]

# ✔ CHECK — compare stage codes against the REAL raster.
#   The real file also has code 8 (small patch), applied after the stages; that is
#   added by hand in section 6, so exclude code-8 pixels from this comparison.
with rasterio.open(OUTDIR / "reason_codes.tif") as src:
    real = src.read(1)

mism = (reason != real) & (real != 8)
print("stage-stage mismatches vs real raster:", f"{int(mism.sum()):,}")
assert mism.sum() == 0, "hand classifier differs from the real staged classifier"
print("✔ staged classifier reproduced (outside code 8 — handled in §6)")

## 6. Small-patch logic *(manual §8.3)*

Label the eligible mask, kill labels smaller than `min_patch_ha` worth of pixels,
re-code them **8**. Compare against the real code-8 pixels (**207.72 ha**).

In [ ]:
from scipy import ndimage
import math

conn = d["patch_connectivity"]
structure = np.ones((3, 3), dtype=int) if conn == 8 else None
min_px = math.ceil(d["min_patch_ha"] / px_ha)
print(f"connectivity={conn} | {d['min_patch_ha']} ha = {min_px} px (ceil division)")

reason_hand = reason.copy()
lab, n = ndimage.label(reason_hand == 0, structure=structure)
sizes = np.bincount(lab.ravel())
print(f"pre-removal patches: {n:,} | size px min/median/max: "
      f"{sizes[1:].min()} / {int(np.median(sizes[1:]))} / {sizes[1:].max()}")

small = sizes < min_px
small[0] = False                          # label 0 = background
newly_small = small[lab] & (reason_hand == 0)
reason_hand[newly_small] = 8
removed_ha = int(newly_small.sum()) * px_ha
print(f"patch-removal removed: {removed_ha:,.2f} ha")

# ✔ CHECK
assert abs(removed_ha - 207.72) < 1.0, f"got {removed_ha:.2f} vs 207.72"
assert (reason_hand == real).all(), "hand full raster is not identical to real reason_codes.tif"
print("✔ hand patch pass + staged classifier == real reason_codes.tif, byte-for-byte")

## 7. Confidence flags *(flags are not exclusions)*

Flags ride on **candidates only**: `SHRUBLAND=1` (WorldCover 20), `LOSS_10_20_YRS=2`,
`CROPLAND_KEPT=4`. In the real file, outside-boundary pixels carry **255**, not 0 —
a detail easy to miss until your byte-for-byte check fails.

In [ ]:
lo, hi = s["flag_loss_window_years"]
flags = np.zeros(inside.shape, dtype=np.uint8)
flags[L["worldcover"] == w["shrubland_worldcover_class"]] |= 1
flags[(loss_year > 0) & (age > lo) & (age <= hi)] |= 2
if not d["exclude_cropland"][cfg["project"]["eligibility_mode"]]:
    flags[L["worldcover"] == d["cropland_worldcover_class"]] |= 4
flags[reason_hand != 0] = 0               # flags describe candidate land only
flags[reason_hand == OUTSIDE] = 255       # outside is 255 in the real file

with rasterio.open(OUTDIR / "confidence_flags.tif") as src:
    real_flags = src.read(1)

flagged_ha = int(((flags > 0) & (flags != 255)).sum()) * px_ha
print(f"flagged candidate: {flagged_ha:,.1f} ha")

# ✔ CHECK — byte-for-byte
assert np.array_equal(flags, real_flags), "flags differ from confidence_flags.tif"
print("✔ hand flags == real confidence_flags.tif exactly")

## 8. Cartography from real data *(manual §9.1)*

Render the real reason raster with your own RGBA color table, using
`extent=(left, right, bottom, top)` — **that exact order** — then *deliberately*
draw the wrong order so your eyes learn how silent the bug is.

In [ ]:
import matplotlib
matplotlib.use("Agg")     # headless-safe; inline still shows via plt.show() in Jupyter
import matplotlib.pyplot as plt
from matplotlib import colors as mcolors
from matplotlib.patches import Patch

REASON_COLORS = {0: "#2E7D32", 1: "#A0522D", 2: "#9E9E9E", 5: "#8D6E63",
                 6: "#7B1FA2", 8: "#FFD54F"}
LABELS = {0: "Candidate (code 0)", 1: "Woody", 2: "Built/water/wetland",
          5: "Steep", 6: "Protected", 8: "Small patch"}

with rasterio.open(OUTDIR / "reason_codes.tif") as src:
    left, bottom, right, top = src.bounds
    reason_vis = src.read(1)

rgba = np.zeros((*reason_vis.shape, 4), dtype="float32")
for code_, color in REASON_COLORS.items():
    m = reason_vis == code_
    if m.any():
        rgba[m] = mcolors.to_rgba(color)
rgba[reason_vis == 255] = (1, 1, 1, 0)         # outside: fully transparent

fig, ax = plt.subplots(figsize=(10, 9))
ax.imshow(rgba, extent=(left, right, bottom, top), origin="upper")   # ← CORRECT order
district.boundary.plot(ax=ax, color="k", linewidth=0.7)
ax.set_title("Hand-built reason codes — Jhabua\n(label 0 is Candidate, never Eligible)")
ax.set_xlabel("UTM 43N easting (m)"); ax.set_ylabel("northing (m)")
ax.legend(handles=[Patch(facecolor=REASON_COLORS[c], label=LABELS[c])
                   for c in sorted(REASON_COLORS)], loc="lower left", fontsize=8)
fig.savefig(LAB / "02_reason_codes_by_hand.png", dpi=200)
plt.show()
print("✔ saved", LAB / "02_reason_codes_by_hand.png")

### 8.1 — The silent `extent` bug *(manual lesson 9.1 Try Yourself)*

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.imshow(rgba, extent=(left, bottom, right, top), origin="upper")   # ← WRONG order
ax.set_title("WRONG: extent=(left, bottom, right, top)\n→ blank/mirrored axes, no traceback")
fig.savefig(LAB / "02_wrong_extent_bug.png", dpi=150)
plt.show()
print("✔ saved", LAB / "02_wrong_extent_bug.png", "— compare with the correct render above.")
print("The bug is silent; that is why the manual insists on (left, right, bottom, top).")

## 9. Scaling — windowed reads *(manual §11.1)*

In [ ]:
from rasterio.windows import from_bounds
from rasterio.enums import Resampling

bounds = district.total_bounds               # (minx, miny, maxx, maxy)
with rasterio.open(ALIGNED / "worldcover_2021.tif") as src:
    win = from_bounds(*bounds, transform=src.transform).round_offsets().round_lengths()
    full = src.read(1, window=win)
    small = src.read(1, window=win,
                     out_shape=(max(1, win.height // 8), max(1, win.width // 8)),
                     resampling=Resampling.nearest)
print(f"windowed district read: {full.shape} = {full.nbytes / 1e6:.1f} MB")
print(f"downsampled read:       {small.shape} = {small.nbytes / 1e6:.2f} MB "
      f"({small.nbytes / full.nbytes:.1%} of window bytes)")
print("`round_offsets/round_lengths` keeps you snapped to the grid — and the downsampled",
      "product is presentation-only, never an analytic shortcut (manual §11.1 caution).")

## 10. Capstone — the assert battery *(manual §12.1)*

In [ ]:
def check(name, cond):
    print(("✔ PASS  " if cond else "✗ FAIL  ") + name)
    assert cond, name

check("grid 3130 × 2933", inside.shape == (3130, 2933))
check("pixel area 0.09 ha", abs(px_ha - 0.09) < 1e-9)
check("district 3,815,340 px", (reason_hand != OUTSIDE).sum() == 3_815_340)
check("candidate (code 0) = 251,292.78 ha",
      abs((reason_hand == 0).sum() * px_ha - 251_292.78) < 1.0)
check("Stage A eligible = 324,093.96 ha",
      abs((~np.isin(reason, [1, 4, 7]) & (reason != OUTSIDE)).sum() * px_ha - 324_093.96) < 1.0)
check("protected = 68,149.08 ha", abs((reason_hand == 6).sum() * px_ha - 68_149.08) < 1.0)
check("hand full raster == real reason_codes.tif", (reason_hand == real).all())
check("hand flags == real confidence_flags.tif", np.array_equal(flags, real_flags))
print("\nAll eight ✔ PASS → you reproduced the full Jhabua screen with your own code.")
print("That is Module 12 of the manual — now read src/ and see what you disagreed with.")

---

### Where to go next

- Tweak one config value in a scratch copy of `config/jhabua.yaml`, re-run one
  section, and watch which spatial rule restores which area (manual §6.3 / §8.1).
- Build the WDPA-outline-only cartographic variant (manual §9.2 Try Yourself).
- Add **one pytest** guarding a rule you now own end-to-end (manual §11.2).
- Then close this notebook and re-read manual §12 — can you tick all seven boxes?

*Note:* hand implementations here intentionally mirror `src/screening.py` v1.1
semantics: stage orders, `source: either`, `shrubland: flag`, flags with outside=255,
small-patch code 8 applied last.